# Run AERMOD in your browser

This notebook builds an AERMOD input file with pyaermod, runs AERMOD on it and maps the
result, all inside this browser tab: nothing is installed on your computer and nothing is
sent to a server.

> **Not for regulatory use.** AERMOD here is EPA's public-domain source compiled to WebAssembly by pyaermod, not EPA's executable. Every change to it is checked against EPA's published AERTEST results (see the second notebook), but use EPA's executable for permit or compliance modeling.

In [ ]:
# Installs into this browser tab only; takes a few seconds the first time.
%pip install -q pyaermod aermod_wasm matplotlib

## 1. Describe the model run

One 50 m stack, a 4 km square grid of receptors, and four days of meteorology from
Albany, NY (March 1988), the data EPA ships with its AERMOD test cases. The files are in
the `data` folder beside this notebook.

In [ ]:
from pyaermod import (
    AERMODProject, CartesianGrid, ControlPathway, MeteorologyPathway, OutputPathway,
    PointSource, PollutantType, ReceptorPathway, SourcePathway, TerrainType,
)

control = ControlPathway(
    title_one="pyaermod in the browser",
    title_two="One stack, four days of Albany 1988 met",
    pollutant_id=PollutantType.OTHER,
    averaging_periods=["1", "24", "PERIOD"],
    terrain_type=TerrainType.FLAT,
)

sources = SourcePathway()
sources.add_source(PointSource(
    source_id="STACK1",
    x_coord=0.0, y_coord=0.0, base_elevation=0.0,
    stack_height=50.0,     # m
    stack_temp=400.0,      # K
    exit_velocity=15.0,    # m/s
    stack_diameter=2.0,    # m
    emission_rate=1.0,     # g/s
))

receptors = ReceptorPathway()
receptors.add_cartesian_grid(CartesianGrid.from_bounds(
    x_min=-2000, x_max=2000, y_min=-2000, y_max=2000, spacing=100,
))

meteorology = MeteorologyPathway(
    surface_file="data/AERMET2.SFC",
    profile_file="data/AERMET2.PFL",
    surface_station_id=14735,
    upper_air_station_id=14735,
    data_start_year=1988,
    profile_base_elevation=0.0,
)

output = OutputPathway(
    receptor_table=True,
    max_table=True,
    summary_file="browser_demo.sum",
    period_plot_files="browser_demo",   # a PLOTFILE per averaging period
)

project = AERMODProject(control=control, sources=sources, receptors=receptors,
                        meteorology=meteorology, output=output)
project.write("browser_demo.inp")
# The deck, without the grid's 82 lines of zero elevations:
print("\n".join(line for line in project.to_aermod_input().splitlines()
                if " ELEV " not in line and " HILL " not in line))

## 2. Run AERMOD

`AERMODRunner` finds the WebAssembly build of AERMOD that `aermod_wasm` installed. The run
takes a few seconds; the page cannot respond until it finishes.

In [ ]:
from pyaermod import AERMODRunner

result = AERMODRunner().run("browser_demo.inp")
print(result)
print(f"{result.warning_count} warning(s), {result.fatal_count} fatal error(s)")

## 3. Read and map the results

In [ ]:
from pyaermod import read_plotfile

plot_names = output.period_plot_file_names(control.averaging_periods)
grid = read_plotfile(plot_names["PERIOD"]).to_dataframe()
peak = grid.loc[grid["AVERAGE_CONC"].idxmax()]
print(f"Highest 4-day average: {peak['AVERAGE_CONC']:.3g} ug/m3 at ({peak['X']:.0f}, {peak['Y']:.0f}) m")
grid.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

pivot = grid.pivot_table(index="Y", columns="X", values="AVERAGE_CONC")
fig, ax = plt.subplots(figsize=(7, 6))
levels = np.linspace(0, pivot.values.max(), 12)
filled = ax.contourf(pivot.columns, pivot.index, pivot.values, levels=levels, cmap="YlOrBr")
ax.plot(0, 0, marker="^", color="black", markersize=9, label="STACK1")
fig.colorbar(filled, ax=ax, label="4-day average concentration (ug/m3)")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_aspect("equal")
ax.legend(loc="upper right")
ax.set_title("Period average, STACK1")
plt.show()

## 4. Try something

Change the stack height, exit velocity or emission rate in step 1, then run steps 1 to 3
again. Doubling the emission rate doubles every concentration; a taller stack lowers the
peak and moves it downwind.

The files AERMOD wrote (`browser_demo.out`, the `.plt` plot files) are in the file browser
on the left. Right-click one to download it.